# TFT Local Training Pipeline

Trains a TemporalFusionTransformer on GC M1 OHLCV bars, stores the best checkpoint **locally** (no GCS / BigQuery anymore)
and then tests the model on a separate test CSV. The forecasts (several horizons) are appended as extra columns to the test CSV.

## 0. Configuration

In [ ]:
import os
os.environ.pop('GOOGLE_APPLICATION_CREDENTIALS', None)

In [ ]:
import torch
print(torch.__version__)               # sollte auf +cu128 enden
print(torch.cuda.is_available())       # True
print(torch.cuda.get_device_name(0))   # NVIDIA GeForce RTX 5080

In [ ]:
# ===== CONFIGURE THESE =====
from pathlib import Path

SYMBOL = "GC"  # Instrument represented by the training data
DATA_CSV_RELATIVE_PATH = (
    Path("data")
    / "Candelexport_GC_M1_25_11_2018_20_10_2026"
    / "E:\Programierung\Daten\Candelexport_GC_M1_25_11_2018_20_10_2026.csv"
)
DATA_CSV_CANDIDATES = [
    root / DATA_CSV_RELATIVE_PATH
    for root in (Path.cwd(), *Path.cwd().parents)
]
DATA_CSV_PATH = next(
    (path.resolve() for path in DATA_CSV_CANDIDATES if path.is_file()),
    None,
)
if DATA_CSV_PATH is None:
    searched_paths = ", ".join(str(path) for path in DATA_CSV_CANDIDATES)
    raise FileNotFoundError(f"Training CSV not found. Searched: {searched_paths}")

# Training hyperparameters
MAX_EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
HIDDEN_SIZE = 64
ATTENTION_HEAD_SIZE = 4
DROPOUT = 0.1
HIDDEN_CONTINUOUS_SIZE = 32
PATIENCE = 10               # Early stopping patience

# Model architecture
MAX_ENCODER_LENGTH = 60  # Input range bars
TARGET_HORIZON_BARS = 60
MAX_PREDICTION_LENGTH = TARGET_HORIZON_BARS

# Local model storage (replaces the former GCS upload)
MODEL_DIR = Path("models")
MODEL_LATEST_PATH = MODEL_DIR / f"tft_checkpoint_{SYMBOL}_latest.ckpt"
TRAINING_LOG_PATH = MODEL_DIR / "training_log.csv"   # replaces the former BigQuery log

# Test / inference on a separate CSV (section 11)
TEST_CSV_PATH = Path(r"E:\Programierung\Daten\Test_GC_M1_20_10_2025_20_10_2026.csv")
# Output = copy of the test CSV with forecast columns appended.
# Set TEST_OUTPUT_CSV_PATH = TEST_CSV_PATH to write into the original file instead.
TEST_OUTPUT_CSV_PATH = TEST_CSV_PATH.with_name(TEST_CSV_PATH.stem + "_predictions.csv")
PREDICTION_HORIZONS = [1, 5, 15, 30, 60]  # bars ahead (M1 -> minutes); each must be <= MAX_PREDICTION_LENGTH
EXPORT_QUANTILES = [0.5]                  # 0.5 = median forecast; e.g. [0.1, 0.5, 0.9] adds a forecast band
PREDICT_BATCH_SIZE = 512

# Text columns with "%" that are converted to numeric features when loading the CSV
PERCENT_COLUMNS = ["Delta/Volume", "Session Delta %"]

# Use GPU if available
import torch
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Training on: {ACCELERATOR}")
print(f"Symbol: {SYMBOL}, Epochs: {MAX_EPOCHS}")
print(f"DATA_CSV_PATH: {DATA_CSV_PATH}")
print(f"TEST_CSV_PATH: {TEST_CSV_PATH}")

## 1. Install / Import Dependencies

In [ ]:
%pip install --upgrade ipywidgets

In [ ]:
import gc
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss, MultiHorizonMetric
import sys
import subprocess
import json
import shutil

torch.set_float32_matmul_precision("medium")

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
pd.set_option("display.max_columns", None)

## 2. Custom Loss Function

In [ ]:
class DirectionAwareQuantileLoss(MultiHorizonMetric):
    """
    Composite loss: QuantileLoss + direction penalty.
    Penalizes predictions where the predicted direction of change
    disagrees with the actual direction of change.
    """

    def __init__(
        self,
        quantile_weight: float = 0.7,
        direction_weight: float = 0.3,
        quantiles: list = [0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98],
        **kwargs,
    ):
        super().__init__(quantiles=quantiles, **kwargs)
        self.quantile_weight = quantile_weight
        self.direction_weight = direction_weight
        self.quantiles = quantiles

    def loss(self, y_pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        losses = []
        for i, q in enumerate(self.quantiles):
            errors = target - y_pred[..., i]
            q_loss = torch.max((q - 1) * errors, q * errors)
            losses.append(q_loss.unsqueeze(-1))
        quantile_loss = torch.cat(losses, dim=-1).mean(dim=-1)

        median_idx = len(self.quantiles) // 2
        pred_median = y_pred[..., median_idx]

        pred_change = pred_median[:, 1:] - pred_median[:, :-1]
        actual_change = target[:, 1:] - target[:, :-1]

        direction_agreement = torch.tanh(pred_change * 10) * torch.tanh(actual_change * 10)
        direction_penalty = torch.clamp(1.0 - direction_agreement, min=0.0) / 2.0

        pad = torch.zeros_like(direction_penalty[:, :1])
        direction_penalty = torch.cat([pad, direction_penalty], dim=1)

        return self.quantile_weight * quantile_loss + self.direction_weight * direction_penalty

    # --- Needed for prediction ---------------------------------------------------------
    # The network outputs the 7 quantiles directly. Without these overrides pytorch_forecasting
    # falls back to the generic Metric behaviour: point forecast = MEAN of the quantiles and
    # mode="quantiles" = quantiles OF the quantiles. Training (loss()) is not affected.
    def to_prediction(self, y_pred: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        """Point forecast = median quantile."""
        if y_pred.ndim == 3:
            return y_pred[..., len(self.quantiles) // 2]
        return y_pred

    def to_quantiles(self, y_pred: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        """Return the predicted quantiles unchanged."""
        if y_pred.ndim == 2:
            return y_pred.unsqueeze(-1)
        return y_pred

### Data Load from the NQ 8-Range CSV

Each row is one NQ 8-range bar. The CSV is loaded once and reused for both training tasks.

In [ ]:
def load_nq_range_csv(csv_path: str | Path = DATA_CSV_PATH) -> pd.DataFrame:
    """Load all NQ 8-range CSV columns in source Bar order."""
    raw = pd.read_csv(
        csv_path,
        sep=";",
        dtype={
            "Bar": "int64",
            "Open": "float32",
            "High": "float32",
            "Low": "float32",
            "Close": "float32",
            "Volume": "float32",
        },
        encoding="utf-8-sig",
    )
    if raw.empty:
        raise ValueError(f"No rows found in NQ CSV: {csv_path}")

    # Percent columns are exported as text ("59.15%") and would otherwise be silently ignored
    # as non-numeric. Convert them to fractions (59.15% -> 0.5915) so they become model features.
    for col in PERCENT_COLUMNS:
        if col not in raw.columns:
            continue
        text = (
            raw[col].astype(str).str.strip()
            .str.replace("%", "", regex=False)
            .str.replace("'", "", regex=False)
            .str.replace(" ", "", regex=False)
        )
        # Decimal comma only if the value has no decimal point (e.g. "59,15%")
        text = text.where(text.str.contains(".", regex=False), text.str.replace(",", ".", regex=False))
        values = pd.to_numeric(text, errors="coerce")
        n_bad = int((values.isna() & (text != "") & (text.str.lower() != "nan")).sum())
        if n_bad:
            print(f"⚠ {col}: {n_bad:,} values not parseable -> set to 0")
        raw[col] = (values / 100).fillna(0.0).astype("float32")

    required_values = ["Time UTC", "Open", "High", "Low", "Close", "Volume"]
    if raw[required_values].isna().any().any():
        raise ValueError(f"NQ CSV contains missing timestamps or OHLCV values: {csv_path}")

    # Preserve the source sequence: timestamps can occasionally reverse by a few seconds.
    if not raw["Bar"].is_monotonic_increasing:
        raw = raw.sort_values("Bar", kind="stable")

    df = raw.rename(columns={
        "Time UTC": "timestamp",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    })
    df["timestamp"] = pd.to_datetime(
        df["timestamp"], format="%d.%m.%Y %H:%M:%S", errors="raise", utc=True
    )
    df = df.reset_index(drop=True)
    print(f"Loaded {len(df):,} NQ 8-range bars from {df['timestamp'].min()} to {df['timestamp'].max()}")
    return df

In [ ]:
df_raw = load_nq_range_csv()
LOOKBACK_DAYS = (df_raw["timestamp"].max() - df_raw["timestamp"].min()).days
df_raw = df_raw.drop(columns=["Time local"])
df_raw.head()

In [ ]:
df_raw.shape

In [ ]:
df_raw.columns

## 3. Loaded NQ 8-Range Bars

In [ ]:
print(f"CSV source: {DATA_CSV_PATH}")
print(f"Bars loaded: {len(df_raw):,}")
print(f"Date range (UTC): {df_raw['timestamp'].min()} to {df_raw['timestamp'].max()}")

In [ ]:
df_raw.tail()

## 4. Feature Engineering

In [ ]:
def calculate_all_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate all 76 features required for the TFT model.
    Input df must have: timestamp, open, high, low, close, volume.
    Lag windows count NQ range bars; legacy minute-suffixed feature names are retained for compatibility.
    """
    df = df.copy()

    # 1. RETURNS (5 features)
    df['returns_1m'] = df['close'].pct_change(1)
    df['returns_5m'] = df['close'].pct_change(5)
    df['returns_15m'] = df['close'].pct_change(15)
    df['returns_30m'] = df['close'].pct_change(30)
    df['returns_60m'] = df['close'].pct_change(60)

    # 2. PRICE RATIOS (6 features)
    df['high_low_ratio'] = df['high'] / df['low']
    df['close_open_ratio'] = df['close'] / df['open']
    df['high_close_ratio'] = df['high'] / df['close']
    df['low_close_ratio'] = df['low'] / df['close']
    df['upper_shadow'] = (df['high'] - np.maximum(df['open'], df['close'])) / (df['high'] - df['low'] + 1e-10)
    df['lower_shadow'] = (np.minimum(df['open'], df['close']) - df['low']) / (df['high'] - df['low'] + 1e-10)

    # 3. SMA FEATURES (11 features)
    for period in [5, 10, 20, 30]:
        sma = df['close'].rolling(window=period).mean()
        df[f'sma_{period}_slope'] = sma.pct_change()
        df[f'close_to_sma_{period}'] = (df['close'] - sma) / sma

    sma_60 = df['close'].rolling(window=60).mean()
    df['close_to_sma_60'] = (df['close'] - sma_60) / sma_60

    sma_120 = df['close'].rolling(window=120).mean()
    df['sma_120_slope'] = sma_120.pct_change()
    df['close_to_sma_120'] = (df['close'] - sma_120) / sma_120

    # 4. MACD (2 features)
    ema_12 = df['close'].ewm(span=12, adjust=False).mean()
    ema_26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema_12 - ema_26
    signal_line = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_histogram'] = df['macd'] - signal_line

    # 5. VOLATILITY (4 features)
    df['volatility_5'] = df['returns_1m'].rolling(window=5).std()
    df['volatility_10'] = df['returns_1m'].rolling(window=10).std()
    df['volatility_20'] = df['returns_1m'].rolling(window=20).std()
    df['volatility_60'] = df['returns_1m'].rolling(window=60).std()

    # 6. ATR (2 features)
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = np.maximum(high_low, np.maximum(high_close, low_close))
    df['atr_14'] = tr.rolling(window=14).mean()
    df['atr_60'] = tr.rolling(window=60).mean()

    # 7. BOLLINGER BANDS (4 features)
    for period in [20, 60]:
        sma = df['close'].rolling(window=period).mean()
        std = df['close'].rolling(window=period).std()
        df[f'bb_std_{period}'] = std
        df[f'bb_position_{period}'] = (df['close'] - sma) / (2 * std + 1e-10)

    # 8. RSI (3 features)
    def calc_rsi(series, period):
        delta = series.diff()
        gain = delta.where(delta > 0, 0).rolling(window=period).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
        rs = gain / (loss + 1e-10)
        return 100 - (100 / (1 + rs))

    df['rsi_14'] = calc_rsi(df['close'], 14)
    df['rsi_20'] = calc_rsi(df['close'], 20)
    df['rsi_60'] = calc_rsi(df['close'], 60)

    # 9. STOCHASTIC (3 features)
    for period in [14, 60]:
        lowest_low = df['low'].rolling(window=period).min()
        highest_high = df['high'].rolling(window=period).max()
        df[f'stoch_k_{period}'] = 100 * (df['close'] - lowest_low) / (highest_high - lowest_low + 1e-10)
    df['stoch_d_14'] = df['stoch_k_14'].rolling(window=3).mean()

    # 10. RATE OF CHANGE (2 features)
    df['roc_10'] = df['close'].pct_change(10) * 100
    df['roc_20'] = df['close'].pct_change(20) * 100

    # 11. VOLUME INDICATORS (12 features)
    df['volume_change'] = df['volume'].pct_change(1)
    df['volume_change_5m'] = df['volume'].pct_change(5)

    for period in [5, 10, 20, 60]:
        df[f'volume_sma_{period}'] = df['volume'].rolling(window=period).mean()
        df[f'volume_ratio_{period}'] = df['volume'] / (df[f'volume_sma_{period}'] + 1e-10)

    # 12. OBV, VPT, MFI (5 features)
    obv = np.where(df['close'] > df['close'].shift(), df['volume'],
                   np.where(df['close'] < df['close'].shift(), -df['volume'], 0))
    df['obv'] = np.cumsum(obv)
    obv_sma = pd.Series(df['obv']).rolling(window=20).mean()
    df['obv_ratio'] = df['obv'] / (obv_sma + 1e-10)

    df['vpt'] = (df['volume'] * df['close'].pct_change()).cumsum()

    def calc_mfi(df, period):
        typical_price = (df['high'] + df['low'] + df['close']) / 3
        money_flow = typical_price * df['volume']
        positive_flow = money_flow.where(typical_price > typical_price.shift(), 0).rolling(window=period).sum()
        negative_flow = money_flow.where(typical_price < typical_price.shift(), 0).rolling(window=period).sum()
        mfi = 100 - (100 / (1 + positive_flow / (negative_flow + 1e-10)))
        return mfi

    df['mfi_14'] = calc_mfi(df, 14)
    df['mfi_60'] = calc_mfi(df, 60)

    # 13. VWAP RATIOS (2 features)
    vwap_20 = (df['volume'] * df['close']).rolling(window=20).sum() / (df['volume'].rolling(window=20).sum() + 1e-10)
    vwap_60 = (df['volume'] * df['close']).rolling(window=60).sum() / (df['volume'].rolling(window=60).sum() + 1e-10)
    df['close_to_vwap_20'] = (df['close'] - vwap_20) / vwap_20
    df['close_to_vwap_60'] = (df['close'] - vwap_60) / vwap_60

    # 14. TIME FEATURES (10 features)
    df['hour'] = df['timestamp'].dt.hour
    df['minute'] = df['timestamp'].dt.minute
    df['day_of_week'] = df['timestamp'].dt.dayofweek
    df['is_morning'] = ((df['hour'] >= 9) & (df['hour'] < 12)).astype(int)
    df['is_afternoon'] = ((df['hour'] >= 12) & (df['hour'] < 16)).astype(int)

    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['minute_sin'] = np.sin(2 * np.pi * df['minute'] / 60)
    df['minute_cos'] = np.cos(2 * np.pi * df['minute'] / 60)
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # 15. VOLUME LAGS (5 features)
    df['volume_lag_1'] = df['volume'].shift(1)
    df['volume_lag_5'] = df['volume'].shift(5)
    df['volume_lag_15'] = df['volume'].shift(15)
    df['volume_lag_30'] = df['volume'].shift(30)
    df['volume_lag_60'] = df['volume'].shift(60)

    # 16. TARGET VARIABLES (2 features)
    # Keep legacy target names; the 60-step horizon is measured in NQ bars, not minutes.
    df['target_close_60m'] = df['close'].shift(-TARGET_HORIZON_BARS)
    df['target_return_60m'] = df['close'].pct_change(TARGET_HORIZON_BARS).shift(-TARGET_HORIZON_BARS)

    return df

In [ ]:
def remove_highly_correlated_features(df: pd.DataFrame, threshold: float = 0.95):
    """Remove features with pairwise correlation > threshold."""
    exclude_cols = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "open", "high", "low", "close", "volume", "vwap",
    ]
    feature_cols = [
        c for c in df.columns
        if c not in exclude_cols and df[c].dtype in ["float64", "float32", "int64"]
    ]

    df_clean = df[feature_cols].dropna()
    corr_matrix = df_clean.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]

    print(f"Correlation filtering (threshold={threshold}):")
    print(f"  Features before: {len(feature_cols)}")
    print(f"  Features dropped: {len(to_drop)} — {to_drop[:10]}{'...' if len(to_drop) > 10 else ''}")
    print(f"  Features remaining: {len(feature_cols) - len(to_drop)}")

    return df.drop(columns=to_drop), to_drop


def prepare_dataset(df: pd.DataFrame):
    """Run feature engineering and prepare the TFT-ready DataFrame."""
    # Calculate VWAP
    if df['volume'].sum() > 0:
        df["vwap"] = (df["volume"] * (df["high"] + df["low"] + df["close"]) / 3).cumsum() / df["volume"].cumsum()
    else:
        df["vwap"] = (df["high"] + df["low"] + df["close"]) / 3

    # Feature engineering
    df_feat = calculate_all_features(df)

    # Drop rows with NaN from indicators
    target_cols = ["target_close_60m", "target_return_60m"]
    non_target = [c for c in df_feat.columns if c not in target_cols]
    df_feat = df_feat.dropna(subset=non_target)
    df_feat = df_feat.dropna(subset=target_cols)
    df_feat = df_feat.reset_index(drop=True)

    # Correlation filtering
    df_feat, dropped_cols = remove_highly_correlated_features(df_feat, threshold=0.95)

    # Add required columns
    df_feat["time_idx"] = range(len(df_feat))
    df_feat["group"] = "default"

    # Clean infinities
    numeric_cols = df_feat.select_dtypes(include=[np.number]).columns.tolist()
    df_feat[numeric_cols] = df_feat[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df_feat[numeric_cols] = df_feat[numeric_cols].ffill().bfill()

    print(f"\nPrepared dataset: {len(df_feat):,} rows, {len(df_feat.columns)} columns")
    return df_feat

In [ ]:
# Run feature engineering
df_tft = prepare_dataset(df_raw)

# Save metadata
data_start = str(df_raw['timestamp'].min())
data_end = str(df_raw['timestamp'].max())
print(f"Data range: {data_start} to {data_end}")

# Free raw data
del df_raw
gc.collect()

In [ ]:
df_tft.head()

## 5. Build TimeSeriesDataSet

In [ ]:
def build_feature_lists(df: pd.DataFrame):
    """Derive the time-varying known and unknown reals lists."""
    time_varying_known_reals = ["hour_sin", "hour_cos", "minute_sin", "minute_cos", "day_sin", "day_cos"]
    time_varying_known_reals = [c for c in time_varying_known_reals if c in df.columns]

    exclude = [
        "timestamp", "time_idx", "group",
        "target_close_60m", "target_return_60m",
        "hour", "minute", "day_of_week",
        "is_morning", "is_afternoon",
    ] + time_varying_known_reals

    time_varying_unknown_reals = [
        c for c in df.columns
        if c not in exclude and pd.api.types.is_numeric_dtype(df[c])
    ]

    if "close" in time_varying_unknown_reals:
        time_varying_unknown_reals.remove("close")
    time_varying_unknown_reals = ["close"] + time_varying_unknown_reals

    print(f"Time-varying known reals: {len(time_varying_known_reals)}")
    print(f"Time-varying unknown reals: {len(time_varying_unknown_reals)}")
    return time_varying_known_reals, time_varying_unknown_reals


time_varying_known_reals, time_varying_unknown_reals = build_feature_lists(df_tft)

# Train/validation split (80/20)
training_cutoff = int(len(df_tft) * 0.8)

print(f"\nDataset size: {len(df_tft):,} rows")
print(f"Training cutoff: time_idx={training_cutoff}")
print(f"Training rows: {len(df_tft[df_tft['time_idx'] <= training_cutoff]):,}")
print(f"Validation rows: {len(df_tft[df_tft['time_idx'] > training_cutoff]):,}")

In [ ]:
training = TimeSeriesDataSet(
    df_tft[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",
    target="close",
    group_ids=["group"],
    min_encoder_length=MAX_ENCODER_LENGTH // 2,
    max_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=["group"],
    time_varying_known_categoricals=[],
    time_varying_known_reals=time_varying_known_reals,
    time_varying_unknown_categoricals=[],
    time_varying_unknown_reals=time_varying_unknown_reals,
    target_normalizer=GroupNormalizer(
        groups=["group"],
        transformation="softplus",
    ),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    df_tft[lambda x: x.time_idx > training_cutoff],
    predict=False,
)

training_rows = len(training)
validation_rows = len(validation)
print(f"Training samples: {training_rows:,}, Validation samples: {validation_rows:,}")

# Free DataFrame
del df_tft
gc.collect()

# Create dataloaders
train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0)
val_dataloader = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)

## 6. Build & Train Model

In [ ]:
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=LEARNING_RATE,
    hidden_size=HIDDEN_SIZE,
    attention_head_size=ATTENTION_HEAD_SIZE,
    dropout=DROPOUT,
    hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
    output_size=7,  # 7 quantiles
    loss=DirectionAwareQuantileLoss(quantile_weight=0.7, direction_weight=0.3),
    reduce_on_plateau_patience=4,
)
print(f"Model parameters: {tft.size() / 1e3:.1f}k")

In [ ]:
# Callbacks
checkpoint_dir = "checkpoints/training"
os.makedirs(checkpoint_dir, exist_ok=True)

early_stop = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=PATIENCE, verbose=True, mode="min")
lr_monitor = LearningRateMonitor(logging_interval="step")
checkpoint_cb = ModelCheckpoint(
    dirpath=checkpoint_dir,
    monitor="val_loss",
    filename=f"tft-{SYMBOL}" + "-{epoch:02d}-{val_loss:.4f}",
    save_top_k=1,
    mode="min",
)

# Trainer
trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    callbacks=[early_stop, lr_monitor, checkpoint_cb],
    enable_model_summary=True,
    accelerator=ACCELERATOR,
    gradient_clip_val=0.1,
)

print(f"Starting training on {ACCELERATOR}...")
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

In [ ]:
best_model_path = checkpoint_cb.best_model_path
best_val_loss = float(checkpoint_cb.best_model_score)
print(f"Best model: {best_model_path}")
print(f"Best val_loss: {best_val_loss:.4f}")
print(f"Epochs trained: {trainer.current_epoch + 1}")

## 7. Evaluate on Validation Set

In [ ]:
# Load best checkpoint
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path, strict=False)

# Get predictions on validation set
predictions = best_tft.predict(val_dataloader, return_x=True)

if predictions.output.dim() == 3:
    pred_values = predictions.output[:, :, 3]  # median quantile
else:
    pred_values = predictions.output

actuals = predictions.x["decoder_target"]
if actuals.dim() == 3:
    actuals = actuals[:, :, 0]

pred_np = pred_values.detach().cpu().numpy().flatten()
actual_np = actuals.detach().cpu().numpy().flatten()

# Metrics
errors = actual_np - pred_np
mae = float(np.mean(np.abs(errors)))
rmse = float(np.sqrt(np.mean(errors ** 2)))
mask = actual_np != 0
mape = float(np.mean(np.abs(errors[mask] / actual_np[mask])) * 100) if mask.any() else None
ss_res = np.sum(errors ** 2)
ss_tot = np.sum((actual_np - np.mean(actual_np)) ** 2)
r_squared = float(1 - ss_res / ss_tot) if ss_tot != 0 else None

print(f"Validation Metrics:")
print(f"  MAE:   {mae:.4f}")
print(f"  RMSE:  {rmse:.4f}")
print(f"  MAPE:  {mape:.4f}%" if mape else "  MAPE:  N/A")
print(f"  R²:    {r_squared:.4f}" if r_squared else "  R²:    N/A")

## 8. Save Model Locally

Replaces the former GCS upload: the best checkpoint is copied to `MODEL_DIR` as `..._latest.ckpt` plus a timestamped copy.

In [ ]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
timestamped_model_path = MODEL_DIR / f"tft_checkpoint_{SYMBOL}_{run_timestamp}.ckpt"

shutil.copy2(best_model_path, MODEL_LATEST_PATH)
shutil.copy2(best_model_path, timestamped_model_path)

print(f"✓ Saved: {MODEL_LATEST_PATH.resolve()}")
print(f"✓ Saved: {timestamped_model_path.resolve()}")

## 9. Log Metrics Locally

Replaces the former BigQuery log: one row per training run is appended to `TRAINING_LOG_PATH` (`;`-separated CSV).

In [ ]:
row = {
    "run_timestamp": datetime.now().isoformat(timespec="seconds"),
    "symbol": SYMBOL,
    "best_val_loss": round(best_val_loss, 6),
    "mae": round(mae, 6),
    "rmse": round(rmse, 6),
    "mape": round(mape, 6) if mape else None,
    "r_squared": round(r_squared, 6) if r_squared else None,
    "epochs_trained": trainer.current_epoch + 1,
    "max_epochs": MAX_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "hidden_size": HIDDEN_SIZE,
    "attention_head_size": ATTENTION_HEAD_SIZE,
    "dropout": DROPOUT,
    "patience": PATIENCE,
    "lookback_days": LOOKBACK_DAYS,
    "data_start_date": data_start,
    "data_end_date": data_end,
    "training_rows": training_rows,
    "validation_rows": validation_rows,
    "model_path": str(timestamped_model_path.resolve()),
}

pd.DataFrame([row]).to_csv(
    TRAINING_LOG_PATH,
    mode="a",
    header=not TRAINING_LOG_PATH.exists(),
    index=False,
    sep=";",
)
print(f"✓ Metrics appended to {TRAINING_LOG_PATH.resolve()}")

## 10. Training Summary

In [ ]:
print("=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print(f"  Symbol:         {SYMBOL}")
print(f"  Epochs:         {trainer.current_epoch + 1}")
print(f"  Best val_loss:  {best_val_loss:.4f}")
print(f"  MAE:            {mae:.4f}")
print(f"  R²:             {r_squared:.4f}" if r_squared else "  R²:  N/A")
print(f"  Data range:     {data_start} → {data_end}")
print(f"  Model:          {MODEL_LATEST_PATH.resolve()}")
print("=" * 60)

## 11. Test on Separate CSV

Applies the saved model to `TEST_CSV_PATH`. For **every bar** the model gets the previous `MAX_ENCODER_LENGTH` bars and
forecasts the close for the next bars; the horizons in `PREDICTION_HORIZONS` are written as columns behind the original
columns (`Pred_Close_+5` = forecast made at this bar for the close 5 bars later; with M1 data = 5 minutes, session gaps count as 0).

* The first ~180 rows stay empty (indicator warm-up incl. SMA-120 + encoder length).
* The feature columns are taken from the checkpoint, so the correlation filter is **not** re-run on the test data.
* **Testing without retraining** (e.g. after a kernel restart): run sections 0, 1 (imports), 2, the loader definition
  in "Data Load", the two definition cells in section 4 – then jump straight here. The model is loaded from `MODEL_LATEST_PATH`.

In [ ]:
# ---- 11.1 Load model -------------------------------------------------------------------
MODEL_PATH_FOR_TEST = MODEL_LATEST_PATH  # or any other .ckpt file

test_model = TemporalFusionTransformer.load_from_checkpoint(
    str(MODEL_PATH_FOR_TEST),
    strict=False,
    map_location="cuda" if torch.cuda.is_available() else "cpu",
)
test_model.eval()

dataset_params = test_model.dataset_parameters
model_quantiles = [float(q) for q in test_model.loss.quantiles]
test_encoder_length = dataset_params["max_encoder_length"]
test_prediction_length = dataset_params["max_prediction_length"]

# Columns the model expects (internal columns are created by TimeSeriesDataSet itself)
INTERNAL_COLS = {"relative_time_idx", "encoder_length"}
model_feature_cols = [
    c for c in dict.fromkeys(dataset_params["time_varying_known_reals"] + dataset_params["time_varying_unknown_reals"])
    if c not in INTERNAL_COLS
]

bad_horizons = [h for h in PREDICTION_HORIZONS if not 1 <= h <= test_prediction_length]
if bad_horizons:
    raise ValueError(f"Horizons {bad_horizons} outside 1..{test_prediction_length}")
quantile_idx = {}
for q in EXPORT_QUANTILES:
    hits = [i for i, mq in enumerate(model_quantiles) if abs(mq - q) < 1e-6]
    if not hits:
        raise ValueError(f"Quantile {q} not in model quantiles {model_quantiles}")
    quantile_idx[q] = hits[0]
median_idx = model_quantiles.index(0.5)

# Features that depend on where the CSV starts -> different meaning in a shorter test file
start_dependent = [c for c in ["Bar", "vwap", "obv", "vpt"] if c in model_feature_cols]
if start_dependent:
    print(f"⚠ Model uses start-dependent features {start_dependent}: their values in the test CSV are not "
          f"comparable to training (counter / cumulative sums restart at the beginning of the file).")

print(f"Model:      {MODEL_PATH_FOR_TEST}")
print(f"Encoder:    {test_encoder_length} bars, horizon: {test_prediction_length} bars")
print(f"Quantiles:  {model_quantiles}")
print(f"Features:   {len(model_feature_cols)}")

In [ ]:
# ---- 11.2 Load test CSV and build features -----------------------------------------------
def build_inference_frame(df: pd.DataFrame, feature_cols: list[str], pad_bars: int) -> pd.DataFrame:
    """
    Same feature pipeline as prepare_dataset(), but
      * no correlation filter (feature list comes from the checkpoint),
      * rows without a future target are kept (we want forecasts up to the last bar),
      * `pad_bars` flat placeholder bars are appended so the decoder of the last real bars has
        timestamps for the known time features. They never receive a forecast themselves.
    """
    df = df.copy().reset_index(drop=True)
    df["is_padding"] = False

    last = df.iloc[[-1]]
    step = df["timestamp"].diff().median()
    pad = pd.concat([last] * pad_bars, ignore_index=True)
    offsets = np.arange(1, pad_bars + 1)
    pad["timestamp"] = df["timestamp"].iloc[-1] + step * offsets
    pad["Bar"] = df["Bar"].iloc[-1] + offsets
    for col in ["open", "high", "low"]:
        pad[col] = last["close"].iloc[0]
    pad["is_padding"] = True
    df = pd.concat([df, pad], ignore_index=True)

    if df["volume"].sum() > 0:
        df["vwap"] = (df["volume"] * (df["high"] + df["low"] + df["close"]) / 3).cumsum() / df["volume"].cumsum()
    else:
        df["vwap"] = (df["high"] + df["low"] + df["close"]) / 3

    df_feat = calculate_all_features(df)

    missing = [c for c in feature_cols if c not in df_feat.columns]
    if missing:
        raise ValueError(f"Test CSV is missing feature columns the model was trained on: {missing}")

    df_feat = df_feat.dropna(subset=feature_cols).reset_index(drop=True)

    numeric_cols = [c for c in df_feat.select_dtypes(include=[np.number]).columns if c != "Bar"]
    df_feat[numeric_cols] = df_feat[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df_feat[numeric_cols] = df_feat[numeric_cols].ffill().bfill()

    return df_feat.assign(time_idx=np.arange(len(df_feat), dtype="int64"), group="default")


# Original file as text -> written back unchanged, only forecast columns are added
test_csv_text = pd.read_csv(TEST_CSV_PATH, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)

df_test_raw = load_nq_range_csv(TEST_CSV_PATH).drop(columns=["Time local"], errors="ignore")
df_inf = build_inference_frame(df_test_raw, model_feature_cols, pad_bars=test_prediction_length)
n_real_rows = int((~df_inf["is_padding"]).sum())

print(f"Test bars:         {len(df_test_raw):,}  ({df_test_raw['timestamp'].min()} → {df_test_raw['timestamp'].max()})")
print(f"After warm-up:     {n_real_rows:,} real rows (+{test_prediction_length} padding rows)")

In [ ]:
# ---- 11.3 Rolling forecast for every bar ----------------------------------------------------
# min_encoder_length = max_encoder_length -> exactly one sample per bar, always with a full history window
test_dataset = TimeSeriesDataSet.from_parameters(
    dataset_params,
    df_inf,
    predict=False,
    stop_randomization=True,
    min_encoder_length=test_encoder_length,
)
test_dataloader = test_dataset.to_dataloader(train=False, batch_size=PREDICT_BATCH_SIZE, num_workers=0)
print(f"Forecast samples: {len(test_dataset):,}")

test_predictions = test_model.predict(
    test_dataloader,
    mode="quantiles",
    return_index=True,
    trainer_kwargs=dict(accelerator=ACCELERATOR, logger=False),
)

q_out = test_predictions.output
if q_out.dim() == 2:
    q_out = q_out.unsqueeze(-1)
q_out = q_out.detach().cpu().numpy()                                   # (samples, horizon, quantiles)
origin_idx = test_predictions.index["time_idx"].to_numpy() - 1          # last bar the model has seen
assert not df_inf.loc[origin_idx, "is_padding"].any()
print(f"Forecast array: {q_out.shape}")

In [ ]:
# ---- 11.4 Append forecast columns and write CSV ---------------------------------------------
pred_df = pd.DataFrame({"_bar": df_inf.loc[origin_idx, "Bar"].to_numpy().astype("int64")})
pred_cols = []
for h in PREDICTION_HORIZONS:
    for q in EXPORT_QUANTILES:
        col = f"Pred_Close_+{h}" if q == 0.5 else f"Pred_Close_+{h}_q{round(q * 100):02d}"
        pred_df[col] = q_out[:, h - 1, quantile_idx[q]].astype("float64").round(2)
        pred_cols.append(col)

test_output = test_csv_text.copy()
test_output["_bar"] = pd.to_numeric(test_output["Bar"]).astype("int64")
test_output = test_output.merge(pred_df, on="_bar", how="left", validate="many_to_one").drop(columns="_bar")

test_output.to_csv(TEST_OUTPUT_CSV_PATH, sep=";", index=False, encoding="utf-8-sig")
filled = test_output[pred_cols[0]].notna().sum()
print(f"✓ Written: {TEST_OUTPUT_CSV_PATH}")
print(f"  {len(test_output):,} rows, {filled:,} with forecast, new columns: {pred_cols}")
test_output[["Bar", "Time UTC", "Close"] + pred_cols].tail(5)

### 11.5 Quick Check Against Actual Prices

For each horizon: model error vs. the naive forecast "price stays where it is", plus the share of correctly predicted directions
(bars without price change are ignored). Note: if the test period overlaps the validation period used for early stopping,
these numbers are somewhat optimistic.

In [ ]:
close_arr = df_inf["close"].to_numpy(dtype="float64")
eval_rows = []
for h in PREDICTION_HORIZONS:
    target_idx = origin_idx + h
    ok = target_idx < n_real_rows                     # actual price must exist in the test file
    pred = q_out[ok, h - 1, median_idx].astype("float64")
    actual = close_arr[target_idx[ok]]
    now = close_arr[origin_idx[ok]]
    moved = actual != now
    eval_rows.append({
        "Horizon (bars)": h,
        "Samples": int(ok.sum()),
        "MAE model": np.mean(np.abs(pred - actual)),
        "MAE naive (no change)": np.mean(np.abs(now - actual)),
        "Direction correct %": 100 * np.mean(np.sign(pred - now)[moved] == np.sign(actual - now)[moved]),
    })

pd.DataFrame(eval_rows).round(3)